# មេរៀន ០៨ - លំនាំរចនាប័ទ្មភ្នាក់ងារច្រើន


## ការតំឡើង


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ហេតុអ្វីបានជា ប្រព័ន្ធភ្នាក់ងារច្រើន?

ការងារពិតនៅពិភពលោកដូចជា ការរៀបចំដំណើរកំសាន្ត មានជំនាញច្រើនប្រភេទផ្សេងៗគ្នា — លូជីស្ទិក​​ ការចេះដឹងក្នុងតំបន់ ការប្រាក់ និងច្រើនទៀត។ ភ្នាក់ងារតែមួយដែលព្យាយាមគ្រប់គ្រងអ្វីៗគ្រប់យ៉ាងជារហ័សនឹងក្លាយទៅជារឿងលំបាក។

ប្រព័ន្ធភ្នាក់ងារច្រើនដោះស្រាយវិញតាមរយៈ **ភាពឯកទេស**៖ ភ្នាក់ងារ​នីមួយៗផ្តោតលើមុខវិជ្ជាដែលមានជំនាញមួយ ដើម្បីផលិតលទ្ធផលដែលមានគុណភាពខ្ពស់ជាងអ្នកទូទៅ។ ពួកគេក៏បង្កើន **សមត្ថភាពបន្ថែម** — អ្នកអាចបន្ថែមភ្នាក់ងារថ្មី (ឧ. អ្នកជំនាញស្រយាលពីចរាចរណ៍ហោះហើរ អ្នកវិភាគភោជនីយដ្ឋាន) ដោយមិនចាំបាច់សរសេរវិញកូដបែបធ្វើការ។ ភ្នាក់ងារនៅក្នុងប្រព័ន្ធបានរួមបញ្ចូលគ្នាតាមរយៈបណ្តាញដែលមានរចនាសម្ព័ន្ធ ដើម្បីបញ្ជូនបរិបទពីមួយទៅមួយទៀត។


## ការបង្កើតភ្នាក់ងារប្រភេទជាពិសេស


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## កសាងផ្លូវការជាស្វីច

`WorkflowBuilder` អនុញ្ញាតឱ្យអ្នកភ្ជាប់តំណាងទៅក្នុងក្រាហ្វដែលមានទិស។ នៅទីនេះយើងបង្កើតផ្លូវការងារងាយស្រួលមានពីរដំណាក់កាលៈ **TravelPlanner** រៀបចំផែនការធ្វើដំណើរ បន្ទាប់មក **TravelConcierge** ពិនិត្យនិងបង្កើនគុណភាពរបស់វា។


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## បន្ថែមភ្នាក់ងារបន្ថែមចូលទៅកាន់ដំណើរការងារ

មួយក្នុងចំណោមអត្ថប្រយោជន៍ធំៗនៃគំរូភ្នាក់ងារច្រើន គឺភាពងាយស្រួលក្នុងការពង្រីក។ ខាងក្រោម យើងបានបន្ថែមភ្នាក់ងារ **BudgetReviewer** ដែលពិនិត្យផែនការប្រឆាំងនឹងថវិកានៃអ្នកធ្វើដំណើរ សញ្ញារិះគន់វត្ថុដែលអាចធ្វើឱ្យចំណាយលើកំណត់ ហើយផ្ដល់អភិបាលដែលជួយសន្សំប្រាក់។ ដំណើរការងារឥឡូវនេះដំណើរការភ្នាក់ងារបានបីនាក់ជាដំណកដំណើរ៖

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## សង្ខេប

នៅក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀប:

1. **បង្កើតភ្នាក់ងារពិសេស** — ម្នាក់មួយមានតួនាទីផ្តោត (ផែនការ, អ្នកសេវាកម្ម, ការពិនិត្យថវិកា)។
2. **ភ្ជាប់ភ្នាក់ងារចូលក្នុងលំនាំដំណើរការតាមលំដាប់** ដោយប្រើ `WorkflowBuilder` និង `add_edge`។
3. **ចាក់បញ្ចាំងចេញពីបណ្ដាញភ្នាក់ងារច្រើន** ដើម្បីតាមដានថា ភ្នាក់ងារណាកំពុងនិយាយ។
4. **ពង្រីកលំនាំដំណើរ** ដោយបន្ថែមភ្នាក់ងារថ្មីទៅខ្សែសង្វាក់ដោយមិនបំលែងភ្នាក់ងារមុនទេ។

គំរូគំនិតភ្នាក់ងារច្រើនធ្វើឲ្យភ្នាក់ងារនីមួយៗមានភាពសាមញ្ញ ខណៈដែលបង្កើតលទ្ធផលដែលមានភាពសម្បូរបែប និងបានពិនិត្យយ៉ាងម៉ត់ចត់ជាងភ្នាក់ងារតែម្នាក់អាចធ្វើបានពិសេសតែម្នាក់។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
